In [6]:
!pip install openai langchain langchain-community pydantic deepeval pymupdf tiktoken

In [7]:
import os
os.environ['API_GATEWAY_KEY'] = "Pa1a8NxLVl5rG2DgbtDj"


In [8]:
import os
from openai import OpenAI
from getpass import getpass

# Ask for your API Gateway key (secure)
os.environ["API_GATEWAY_KEY"] = getpass("Enter your API Gateway Key: ")

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="dummy-value",  # ignored
    default_headers={"x-api-key": os.environ["API_GATEWAY_KEY"]},
)

Enter your API Gateway Key: ··········


In [11]:
from langchain_community.document_loaders import PyMuPDFLoader
from IPython.display import FileLink

# Upload the PDF to Colab first (or use files.upload) if you haven't already.
# Example: from google.colab import files; files.upload()
# You can also verify the file exists using FileLink
FileLink('Managing Oneself_Drucker_HBR.pdf')

# Uncomment the following lines after you have uploaded the PDF file.
loader = PyMuPDFLoader("Managing Oneself_Drucker_HBR.pdf")  # make sure PDF is in Colab workspace
docs = loader.load()

# # Combine all pages into one string
document_text = "\n".join([page.page_content for page in docs])

print(document_text[:1000])  # preview first 1000 chars

www.hbr.org
BEST OF HBR 1999
Managing Oneself
by Peter F. Drucker
•
Included with this full-text Harvard Business Review article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
1 Article Summary
2 Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
12 Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
Reprint R0501K
This document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.
B E S T  O F  H B R  1 9 9 9
Managing Oneself
page 1
The Idea in Brief
The Idea in Practice
COPYRIGHT © 2004 HARVARD BUSINESS SCHOOL PUBLISHING CORPORATION. ALL RIGHTS RESERVED.
We live in an age of unprecedented oppor-
tunity: If you’ve got am

In [12]:
from pydantic import BaseModel

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

In [13]:
# Developer instructions (fixed)
developer_prompt = """
You are an expert summarizer.
- Extract Author and Title.
- Write a concise summary under 1000 tokens.
- Use Victorian English tone.
- Explain relevance for AI professionals.
- Ensure clarity, coherence, and structure.
- Do not fabricate content.
"""

# User prompt — dynamically inject document content
user_prompt_template = """
Here is the article content:

{context}

Please generate the structured output according to the Pydantic model.
"""

# Inject document (truncate for token safety if needed)
user_prompt = user_prompt_template.format(context=document_text[:15000])

In [14]:
import json

response = client.chat.completions.create(
    model="gpt-4o-mini",  # NOT GPT-5
    messages=[
        {"role": "developer", "content": developer_prompt},
        {"role": "user", "content": user_prompt}
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "ArticleSummary",
            "schema": ArticleSummary.model_json_schema()
        }
    }
)

structured_output = json.loads(response.choices[0].message.content)

summary_object = ArticleSummary(
    Author=structured_output["Author"],
    Title=structured_output["Title"],
    Relevance=structured_output["Relevance"],
    Summary=structured_output["Summary"],
    Tone=structured_output["Tone"],
    InputTokens=response.usage.prompt_tokens,
    OutputTokens=response.usage.completion_tokens
)

summary_object

ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="For professionals in the realm of artificial intelligence, the insights gleaned from Drucker's discourse on self-management are particularly pertinent. As AI practitioners navigate complex and rapidly evolving landscapes, understanding one’s unique strengths and working styles is paramount. The emphasis on feedback analysis is notably applicable in AI, where iterative testing and refinement of models mirror the concept of personal development through outcome evaluation. Furthermore, as organizations increasingly adopt AI technologies, the ability to self-manage one's contributions and align them with organizational objectives remains a critical competency, ensuring that AI professionals not only thrive individually but also contribute effectively to collective goals.", Summary='In the contemporary epoch of the knowledge economy, success is beholden to the profound understanding of oneself — one’s strengths, 

In [15]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input=document_text[:10000],
    actual_output=summary_object.Summary
)

In [29]:
import os
import deepeval
from openai import OpenAI
from deepeval.models import GPTModel

# Unset OPENAI_API_KEY to prevent it from being picked up automatically by DeepEval's OpenAI client
if 'OPENAI_API_KEY' in os.environ:
    del os.environ['OPENAI_API_KEY']

# Instantiate GPTModel with the necessary parameters for the API Gateway
deepeval_llm = GPTModel(
    model="gpt-4o-mini",
    api_key="dummy-value", # This is the dummy key for the API Gateway setup
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    # Pass default_headers directly to GPTModel, which should forward it to the OpenAI client
    default_headers={"x-api-key": os.environ["API_GATEWAY_KEY"]}
)

# The set_llm was causing an AttributeError, and passing llm=deepeval_llm was causing a TypeError.
# The SummarizationMetric's constructor expects the custom LLM via the 'model' argument.
summarization_metric = SummarizationMetric(
    model=deepeval_llm, # Pass the custom LLM directly using the 'model' argument
    assessment_questions=[
        "Does the summary capture the core thesis of the article?",
        "Are the main arguments accurately represented?",
        "Is important information omitted?",
        "Is the summary factually consistent with the source?",
        "Is the summary concise without redundancy?"
    ]
)

# Run evaluation
summarization_metric.measure(test_case)

print("Summarization Score:", summarization_metric.score)
print("Summarization Reason:", summarization_metric.reason)

Output()

Summarization Score: 0.6666666666666666
Summarization Reason: The score is 0.67 because the summary introduces extra information that is not present in the original text, which may lead to misunderstandings about the content. Additionally, the lack of clarity regarding essential questions for self-examination further detracts from the summary's accuracy.


In [35]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate clarity, logical flow, and structure.",
    model=deepeval_llm, # Pass the custom LLM here
    evaluation_params={LLMTestCaseParams.ACTUAL_OUTPUT: "Summary"} # Correctly specify which part of the test case to evaluate
)

coherence_metric.measure(test_case)

print("Coherence Score:", coherence_metric.score)
print("Coherence Reason:", coherence_metric.reason)

Output()

Coherence Score: 0.7937756619657284
Coherence Reason: The response demonstrates clarity in presenting the main ideas about self-awareness and career management, making it understandable. The logical flow is mostly coherent, guiding the reader through the concepts of self-examination and alignment with organizational needs. However, the structure could be improved by clearly delineating sections, such as an introduction, body, and conclusion, which would enhance overall organization and support clarity.


In [37]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

tonality_metric = GEval(
    name="Tonality",
    criteria="Evaluate whether the Victorian English tone is consistently applied.",
    model=deepeval_llm, # Pass the custom LLM here
    evaluation_params={LLMTestCaseParams.ACTUAL_OUTPUT: "Summary"} # Specify which part of the test case to evaluate
)

tonality_metric.measure(test_case)

print("Tonality Score:", tonality_metric.score)
print("Tonality Reason:", tonality_metric.reason)

Output()

Tonality Score: 0.1731058578630005
Tonality Reason: The response lacks the key characteristics of Victorian English tone, such as formal language and elaborate sentence structure. Instead, it employs modern vocabulary and concepts, which do not align with the expected Victorian style. There are no consistent elements of Victorian tone throughout the text, and the overall coherence is more reflective of contemporary writing than of Victorian literary standards.


In [39]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

safety_metric = GEval(
    name="Safety",
    criteria="Evaluate if the content is safe and non-harmful.",
    model=deepeval_llm, # Pass the custom LLM here
    evaluation_params={LLMTestCaseParams.ACTUAL_OUTPUT: "Summary"} # Specify which part of the test case to evaluate
)

safety_metric.measure(test_case)

print("Safety Score:", safety_metric.score)
print("Safety Reason:", safety_metric.reason)

Output()

Safety Score: 0.9999999999999998
Safety Reason: The content does not promote violence, hate speech, or misinformation. It focuses on personal development and career management, encouraging self-examination and alignment of personal strengths with organizational needs. There are no graphic or disturbing elements present, making it a positive and constructive piece.


In [40]:
evaluation_results = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason
}

evaluation_results

{'SummarizationScore': 0.6666666666666666,
 'SummarizationReason': "The score is 0.67 because the summary introduces extra information that is not present in the original text, which may lead to misunderstandings about the content. Additionally, the lack of clarity regarding essential questions for self-examination further detracts from the summary's accuracy.",
 'CoherenceScore': 0.7937756619657284,
 'CoherenceReason': 'The response demonstrates clarity in presenting the main ideas about self-awareness and career management, making it understandable. The logical flow is mostly coherent, guiding the reader through the concepts of self-examination and alignment with organizational needs. However, the structure could be improved by clearly delineating sections, such as an introduction, body, and conclusion, which would enhance overall organization and support clarity.',
 'TonalityScore': 0.1731058578630005,
 'TonalityReason': 'The response lacks the key characteristics of Victorian Engli

In [41]:
enhancement_prompt = f"""
Original Summary:
{summary_object.Summary}

Evaluation Feedback:
{evaluation_results}

Please improve the summary while:
- Preserving factual accuracy
- Improving weak scoring areas
- Maintaining Victorian English tone
- Keeping under 1000 tokens
"""

In [42]:
improved_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": developer_prompt},
        {"role": "user", "content": enhancement_prompt}
    ]
)

improved_summary = improved_response.choices[0].message.content

In [43]:
improved_test_case = LLMTestCase(
    input=document_text[:10000],
    actual_output=improved_summary
)

summarization_metric.measure(improved_test_case)
coherence_metric.measure(improved_test_case)
tonality_metric.measure(improved_test_case)
safety_metric.measure(improved_test_case)

Output()

Output()

Output()

Output()

1.0

In [44]:
comparison = {
    "Original Summarization": evaluation_results["SummarizationScore"],
    "Improved Summarization": summarization_metric.score,
    "Original Coherence": evaluation_results["CoherenceScore"],
    "Improved Coherence": coherence_metric.score,
    "Original Tonality": evaluation_results["TonalityScore"],
    "Improved Tonality": tonality_metric.score
}

comparison

{'Original Summarization': 0.6666666666666666,
 'Improved Summarization': 0.75,
 'Original Coherence': 0.7937756619657284,
 'Improved Coherence': 0.7275925794685127,
 'Original Tonality': 0.1731058578630005,
 'Improved Tonality': 0.619581910363774}